# Обучение детектора сорняков (Weed Detector)

Этот ноутбук предназначен для обучения модели YOLOv8 для детекции сорняков.
После обучения модель будет использоваться в режиме Green-on-Green.

## Инструкция:
1. Установите зависимости: `pip install ultralytics opencv-python jupyter matplotlib pandas`
2. Подготовьте датасет в формате YOLO (папки images/labels)
3. Запустите все ячейки последовательно
4. Модель будет сохранена в ml/models/weeds_yolo.pt

## 1. Установка зависимостей

In [ ]:
!pip install ultralytics opencv-python matplotlib seaborn pandas -q

## 2. Импорт библиотек

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## 3. Подготовка датасета

Структура папок должна быть следующей:
```
datasets/weeds/
├── dataset.yaml
├── train/
│   ├── images/
│   └── labels/
├── val/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

Формат разметки YOLO: `<class_id> <x_center> <y_center> <width> <height>`

In [ ]:
dataset_dir = Path("datasets/weeds")
train_images_dir = dataset_dir / "train" / "images"
val_images_dir = dataset_dir / "val" / "images"
test_images_dir = dataset_dir / "test" / "images"

print(f"📁 Директория датасета: {dataset_dir.absolute()}")
print(f"📊 Изображений для обучения: {len(list(train_images_dir.glob('*.jpg')) + list(train_images_dir.glob('*.png')))}")
print(f"📊 Изображений для валидации: {len(list(val_images_dir.glob('*.jpg')) + list(val_images_dir.glob('*.png')))}")
print(f"📊 Изображений для тестирования: {len(list(test_images_dir.glob('*.jpg')) + list(test_images_dir.glob('*.png')))}")

## 4. Создание конфигурации датасета

In [ ]:
dataset_config = {
    'path': str(dataset_dir.absolute()),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['weed']
}

config_path = dataset_dir / "dataset.yaml"
with open(config_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"✅ Конфигурация сохранена: {config_path}")
print(yaml.dump(dataset_config, default_flow_style=False))

## 5. Загрузка предобученной модели

In [ ]:
model = YOLO('yolov8n.pt')
print("✅ Модель YOLOv8n загружена")

## 6. Настройка параметров обучения

In [ ]:
training_args = {
    'data': str(config_path),
    'epochs': 50,
    'batch': 16,
    'imgsz': 640,
    'device': 'cpu',
    'workers': 4,
    'optimizer': 'SGD',
    'lr0': 0.01,
    'lrf': 0.1,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'patience': 20,
    'save': True,
    'plots': True,
    'project': 'runs/detect',
    'name': 'weeds_train',
    'exist_ok': True,
    'seed': 42,
}

print("📋 Параметры обучения:")
for key, value in training_args.items():
    print(f"  {key}: {value}")

## 7. Запуск обучения

In [ ]:
print("🚀 Начало обучения...")
results = model.train(**training_args)
print("✅ Обучение завершено!")

## 8. Анализ результатов

In [ ]:
import pandas as pd

results_dir = Path(training_args['project']) / training_args['name']
metrics_csv = results_dir / 'results.csv'

if metrics_csv.exists():
    metrics_df = pd.read_csv(metrics_csv)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    axes[0, 0].plot(metrics_df['epoch'], metrics_df['metrics/precision(B)'], label='Precision')
    axes[0, 0].plot(metrics_df['epoch'], metrics_df['metrics/recall(B)'], label='Recall')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(metrics_df['epoch'], metrics_df['metrics/mAP50(B)'], label='mAP@0.5', color='green')
    axes[0, 1].plot(metrics_df['epoch'], metrics_df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='orange')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    axes[1, 0].plot(metrics_df['epoch'], metrics_df['train/box_loss'], label='Box Loss')
    axes[1, 0].plot(metrics_df['epoch'], metrics_df['train/cls_loss'], label='Class Loss')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    axes[1, 1].plot(metrics_df['epoch'], metrics_df['val/box_loss'], label='Val Box Loss')
    axes[1, 1].plot(metrics_df['epoch'], metrics_df['val/cls_loss'], label='Val Class Loss')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(results_dir / 'training_metrics.png', dpi=150)
    plt.show()
    print(f"📊 Графики сохранены в {results_dir / 'training_metrics.png'}")

## 9. Валидация модели

In [ ]:
print("🔍 Валидация модели...")
val_results = model.val(data=str(config_path), split='val', batch=16, device='cpu', plots=True)

print(f"✅ Метрики валидации:")
print(f"  mAP@0.5: {val_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {val_results.box.map75:.4f}")
print(f"  Precision: {val_results.box.mp:.4f}")
print(f"  Recall: {val_results.box.mr:.4f}")

## 10. Тестирование на примерах

In [ ]:
best_model_path = results_dir / 'weights' / 'best.pt'
test_model = YOLO(str(best_model_path))

test_images = list(test_images_dir.glob('*.jpg'))[:5]

if test_images:
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    
    for idx, img_path in enumerate(test_images):
        if idx >= 5:
            break
        
        results = test_model.predict(source=str(img_path), conf=0.4, verbose=False)
        
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        result_img = results[0].plot()
        result_img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        
        axes[0, idx].imshow(img_rgb)
        axes[0, idx].set_title(f'Original: {img_path.name}')
        axes[0, idx].axis('off')
        
        axes[1, idx].imshow(result_img_rgb)
        axes[1, idx].set_title(f'Detections: {len(results[0].boxes)}')
        axes[1, idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(results_dir / 'test_predictions.png', dpi=150)
    plt.show()
else:
    print("⚠️ Тестовые изображения не найдены")

## 11. Экспорт модели

In [ ]:
import shutil

output_model_path = Path('models/weeds_yolo.pt')
output_model_path.parent.mkdir(parents=True, exist_ok=True)

shutil.copy(best_model_path, output_model_path)

print(f"✅ Модель экспортирована: {output_model_path.absolute()}")
print(f"   Размер модели: {output_model_path.stat().st_size / 1024 / 1024:.2f} MB")

## 12. Рекомендации по улучшению

### Если качество низкое (mAP < 0.5):
1. Увеличьте датасет до 200-500 изображений
2. Добавьте разнообразие условий освещения
3. Проверьте качество разметки
4. Увеличьте epochs до 100
5. Попробуйте yolov8s.pt

### Для ускорения на Orange Pi:
1. Используйте imgsz=416 или 320
2. Включите skip_frames=3
3. Рассмотрите квантование INT8